# 03 — Train GroundZero

**Run on Colab T4 GPU.** Trains the full GroundZero temporal video grounding model.

## What this notebook does
1. Downloads `shaunmarvell/qvhighlights-1fps` from HuggingFace (35 GB)
2. Initialises the model (SigLIP 2 backbone shared — only loaded once)
3. Trains for 20 epochs with AdamW + cosine warmup
4. Logs to W&B every 50 steps
5. Saves best checkpoint to HuggingFace Hub

## Architecture recap
```
frames  → VisualEncoder (SigLIP 2 + LoRA) → TemporalContextModule → ─┐
query   → TextEncoder  (SigLIP 2 shared)  →                           ├→ CrossModalTransformer → SpanHead
                                                                       ┘
```
Total trainable: ~154 M (LoRA 0.15M + TemporalConv 26.6M + CrossAttn 127.5M + SpanHead 0.67M)

## Loss
`total = span_CE + 0.5 * iou_loss + 0.1 * contrastive + conf_bce`


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if not result.stdout:
    raise RuntimeError('No GPU found. Runtime → Change runtime type → T4 GPU')


In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────────
!pip install -q transformers==4.47.0 peft==0.14.0 huggingface_hub datasets wandb einops


In [ ]:
# ── Cell 3: Authenticate ──────────────────────────────────────────────────────
# Store HF_TOKEN and WANDB_KEY in Colab Secrets (left sidebar → key icon)
from google.colab import userdata
from huggingface_hub import login
import wandb

HF_TOKEN   = userdata.get('HF_TOKEN')
WANDB_KEY  = userdata.get('WANDB_KEY')

login(token=HF_TOKEN)
wandb.login(key=WANDB_KEY)
print('Auth OK')


In [ ]:
# ── Cell 4: Clone GroundZero repo ─────────────────────────────────────────────
# Update the URL to your actual repo if needed
!git clone https://github.com/shaunmarvell/groundzero.git /content/groundzero 2>&1 | tail -3
!pip install -q -r /content/groundzero/groundzero-backend/requirements.txt

import sys
sys.path.insert(0, '/content/groundzero/groundzero-backend')
print('Repo loaded')


## Download Dataset

Downloads ~35 GB from `shaunmarvell/qvhighlights-1fps`. First run takes ~30–60 min on Colab.
Subsequent runs in the same session reuse the cache.

> **Note:** The first 1 000 train videos are stored as individual JPEGs in the `frames/` folder
> in the HuggingFace repo (not tarred). Batches 1 000–7 445 are in tar files. All val frames
> are in `val_frames_000000_end.tar`.


In [ ]:
# ── Cell 5: Download annotations (fast, ~2 MB total) ─────────────────────────
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

DATASET_ID = 'shaunmarvell/qvhighlights-1fps'
DATA_DIR   = Path('/content/qvhighlights')
DATA_DIR.mkdir(exist_ok=True)

for fname in ['annotations_train.jsonl', 'annotations_val.jsonl']:
    p = hf_hub_download(DATASET_ID, fname, repo_type='dataset', local_dir=str(DATA_DIR))
    print(f'  {fname} → {p}')

TRAIN_JSONL = DATA_DIR / 'annotations_train.jsonl'
VAL_JSONL   = DATA_DIR / 'annotations_val.jsonl'
FRAMES_DIR  = DATA_DIR / 'frames'
FRAMES_DIR.mkdir(exist_ok=True)

# Quick count
import json
n_train = sum(1 for _ in open(TRAIN_JSONL))
n_val   = sum(1 for _ in open(VAL_JSONL))
print(f'Train: {n_train} annotations, Val: {n_val} annotations')


In [ ]:
# ── Cell 6: Download + extract train tars (batches 1000–7445) ─────────────────
# Skips tars that are already extracted (checks for the vid folder in frames/)
import tarfile

TRAIN_TARS = [
    'frames_001000_002000.tar',
    'frames_002000_003000.tar',
    'frames_003000_004000.tar',
    'frames_004000_005000.tar',
    'frames_005000_006000.tar',
    'frames_006000_007241.tar',
    'frames_007242_007445.tar',
]

for tar_name in TRAIN_TARS:
    print(f'Downloading {tar_name}...')
    tar_path = hf_hub_download(DATASET_ID, tar_name, repo_type='dataset', local_dir=str(DATA_DIR))
    print(f'  Extracting...')
    with tarfile.open(tar_path) as t:
        t.extractall(str(DATA_DIR))  # extracts to frames/{vid}/*.jpg
    os.remove(tar_path)             # free ~4 GB per tar
    # count vids extracted so far
    n_vids = sum(1 for p in FRAMES_DIR.iterdir() if p.is_dir())
    print(f'  Done. Total vid dirs so far: {n_vids}')


In [ ]:
# ── Cell 7: Download + extract val tar ───────────────────────────────────────
print('Downloading val_frames_000000_end.tar...')
tar_path = hf_hub_download(DATASET_ID, 'val_frames_000000_end.tar', repo_type='dataset', local_dir=str(DATA_DIR))
print('  Extracting...')
with tarfile.open(tar_path) as t:
    t.extractall(str(DATA_DIR))
os.remove(tar_path)
print('Val frames extracted.')


In [ ]:
# ── Cell 8: Download first batch (vids 0–1000, stored as individual files) ────
# Uses snapshot_download with allow_patterns — downloads all frames/ files
# that are NOT covered by the tars above. This is the slower step (~5 GB,
# ~150 000 small JPEG files).
from huggingface_hub import snapshot_download

print('Downloading first batch (individual files in frames/ folder)...')
snapshot_download(
    DATASET_ID,
    repo_type='dataset',
    allow_patterns=['frames/**'],
    local_dir=str(DATA_DIR),
    local_dir_use_symlinks=False,
)

n_vids = sum(1 for p in FRAMES_DIR.iterdir() if p.is_dir())
print(f'Total vid dirs: {n_vids}  (train+val, expect ~8995)')


In [ ]:
# ── Cell 9: Verify data ───────────────────────────────────────────────────────
import random
from PIL import Image

# Check a random train annotation has frames on disk
with open(TRAIN_JSONL) as f:
    samples = [json.loads(l) for l in f]

s = random.choice(samples)
vid_dir = FRAMES_DIR / s['vid']
jpgs = sorted(vid_dir.glob('*.jpg'))

print(f"vid:         {s['vid']}")
print(f"query:       {s['query']}")
print(f"gt:          {s['gt_start_sec']:.1f}s → {s['gt_end_sec']:.1f}s")
print(f"n_frames:    {len(jpgs)}")

img = Image.open(jpgs[0])
print(f"frame size:  {img.size}")

assert img.size == (384, 384), 'Expected 384x384 JPEGs'
print('Data OK')


## Training Config


In [ ]:
# ── Cell 10: Config ───────────────────────────────────────────────────────────
CFG = {
    # Data
    'train_jsonl':   str(TRAIN_JSONL),
    'val_jsonl':     str(VAL_JSONL),
    'frames_dir':    str(FRAMES_DIR),

    # Model
    'model_id':      'google/siglip2-so400m-patch14-384',
    'd_model':       1152,
    'n_heads':       8,
    'n_layers':      4,
    'lora_rank':     8,
    'lora_alpha':    16,
    'lora_layers':   [23, 24, 25, 26],
    'dropout':       0.1,

    # Training
    'device':        'cuda',
    'batch_size':    2,      # T4 (16 GB): batch_size=2 is safe
    'lr':            1e-4,
    'weight_decay':  0.01,
    'epochs':        20,
    'warmup_steps':  500,
    'clip_grad':     1.0,
    'log_every':     50,     # W&B log frequency (steps)
    'val_every':     1,      # validate every N epochs

    # Loss weights (total = span + iou_w*iou + cont_w*cont + conf)
    'iou_weight':    0.5,
    'cont_weight':   0.1,

    # Checkpoint
    'ckpt_dir':      '/content/checkpoints',
    'hf_repo':       'shaunmarvell/groundzero-weights',  # your HF model repo
}

Path(CFG['ckpt_dir']).mkdir(exist_ok=True)
print('Config loaded')


## Dataset + DataLoader


In [ ]:
# ── Cell 11: Dataset + DataLoader ────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader
from app.training.dataset import GroundingDataset, collate_fn

train_ds = GroundingDataset(
    jsonl_path=CFG['train_jsonl'],
    frames_root=CFG['frames_dir'],
    augment=True,
)
val_ds = GroundingDataset(
    jsonl_path=CFG['val_jsonl'],
    frames_root=CFG['frames_dir'],
    augment=False,
)

train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'], shuffle=True,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG['batch_size'], shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
    persistent_workers=True,
)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')


## Model Initialisation


In [ ]:
# ── Cell 12: Build model ──────────────────────────────────────────────────────
# SigLIP 2 is loaded ONCE and shared between VisualEncoder + TextEncoder.
# Saves ~1.1 GB VRAM versus the old code that loaded the backbone twice.
from app.pipeline.groundzero_model import GroundZeroModel

model = GroundZeroModel(
    model_id=CFG['model_id'],
    d_model=CFG['d_model'],
    n_heads=CFG['n_heads'],
    n_layers=CFG['n_layers'],
    lora_rank=CFG['lora_rank'],
    lora_alpha=CFG['lora_alpha'],
    lora_layers=CFG['lora_layers'],
    dropout=CFG['dropout'],
    device=CFG['device'],
)

# Print param counts
counts = model.count_params()
print(f"Trainable params: {counts['total_trainable']/1e6:.1f} M")
print(f"Frozen params:    {counts['total_frozen']/1e6:.1f} M")
for k, v in counts['components'].items():
    print(f"  {k:20s} trainable={v['trainable']/1e6:.3f}M  total={v['total']/1e6:.1f}M")


In [ ]:
# ── Cell 13: Optimizer + Scheduler ───────────────────────────────────────────
# AdamW with separate weight decay: LoRA and bias params get no decay.
# Everything else (temporal conv, cross-modal, span head) gets decay=0.01.
from transformers import get_cosine_schedule_with_warmup

decay_params    = []
no_decay_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    # LoRA matrices and all bias terms: no weight decay
    if 'lora_' in name or 'bias' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {'params': decay_params,    'weight_decay': CFG['weight_decay']},
        {'params': no_decay_params, 'weight_decay': 0.0},
    ],
    lr=CFG['lr'],
)

total_steps = len(train_loader) * CFG['epochs']
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CFG['warmup_steps'],
    num_training_steps=total_steps,
)

print(f'Optimizer: AdamW  lr={CFG["lr"]}  decay params={len(decay_params)}  no-decay params={len(no_decay_params)}')
print(f'Scheduler: cosine warmup  warmup={CFG["warmup_steps"]}  total={total_steps} steps')


## Training Step

Each batch contains videos of potentially different lengths (after augmentation).
We pad frame embeddings to N_max within the batch and pass per-sample fractional
timestamps `(B, N_max)` to the temporal context module.


In [ ]:
# ── Cell 14: Encode batch helper ──────────────────────────────────────────────
import torch.nn.functional as F

DEVICE = CFG['device']

def encode_batch(batch):
    """
    Encode frames + queries for a batch. Pads frame embeddings to N_max.

    Returns:
        frame_embs : (B, N_max, 1152)
        timestamps : (B, N_max)   — fractional t/duration, padded with 1.0
        query_emb  : (B, 1, 1152)
    """
    B = len(batch['frames'])
    D = CFG['d_model']

    emb_list = []
    ts_list  = []

    for i in range(B):
        frames_i = batch['frames'][i]                         # List[PIL]
        ts_i     = batch['timestamps'][i]                     # List[float]
        dur_i    = batch['duration'][i].item()

        emb_i = model.visual_encoder.encode_frames(frames_i)  # (N_i, D)
        frac_i = torch.tensor(
            [t / dur_i for t in ts_i],
            dtype=torch.float32, device=DEVICE
        )                                                      # (N_i,)

        emb_list.append(emb_i)
        ts_list.append(frac_i)

    N_max = max(e.shape[0] for e in emb_list)

    frame_embs = torch.zeros(B, N_max, D, device=DEVICE)
    timestamps = torch.ones(B, N_max, device=DEVICE)   # 1.0 = safe default for padding

    for i, (emb, ts) in enumerate(zip(emb_list, ts_list)):
        n = emb.shape[0]
        frame_embs[i, :n] = emb
        timestamps[i, :n] = ts

    query_emb = model.text_encoder.encode_queries(batch['query'])  # (B, D)
    query_emb = query_emb.unsqueeze(1)                             # (B, 1, D)

    return frame_embs, timestamps, query_emb


In [ ]:
# ── Cell 15: Training step ────────────────────────────────────────────────────
from app.training.losses import combined_loss

def train_step(batch):
    """
    One gradient update. Returns dict of loss scalars for W&B logging.
    """
    model.train()

    frame_embs, timestamps, query_emb = encode_batch(batch)

    start_logits, end_logits, confidence = model(
        frame_embs, timestamps, query_emb
    )

    gt_start = batch['gt_start_idx'].to(DEVICE)  # (B,)
    gt_end   = batch['gt_end_idx'].to(DEVICE)    # (B,)
    is_neg   = batch['is_negative'].to(DEVICE)   # (B,)

    # ── Span + IoU + contrastive (positive samples only) ─────────────────────
    pos_mask = ~is_neg
    if pos_mask.any():
        total, span_l, iou_l, cont_l = combined_loss(
            start_logits[pos_mask],
            end_logits[pos_mask],
            gt_start[pos_mask],
            gt_end[pos_mask],
            query_emb[pos_mask],
            frame_embs[pos_mask],
        )
    else:
        total = span_l = iou_l = cont_l = torch.tensor(0.0, device=DEVICE)

    # ── Confidence BCE: positives → target=1, negatives → target=0 ───────────
    conf_targets = torch.where(is_neg, torch.zeros_like(confidence), torch.ones_like(confidence))
    conf_l = F.binary_cross_entropy(confidence, conf_targets)

    loss = total + conf_l

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad],
        CFG['clip_grad']
    )
    optimizer.step()
    scheduler.step()

    return {
        'train/loss':       loss.item(),
        'train/span_loss':  span_l.item(),
        'train/iou_loss':   iou_l.item(),
        'train/cont_loss':  cont_l.item(),
        'train/conf_loss':  conf_l.item(),
        'train/lr':         scheduler.get_last_lr()[0],
    }


## Validation

Validation metric: **R@1 IoU=0.5** — the fraction of queries where the predicted
span overlaps the ground truth by ≥ 50%. Industry standard for temporal grounding.


In [ ]:
# ── Cell 16: Validation function ─────────────────────────────────────────────
from app.pipeline.span_extraction import decode_best_span, to_seconds

def temporal_iou(s1, e1, s2, e2):
    """IoU between two 1-D intervals."""
    inter = max(0.0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)
    return inter / union if union > 0 else 0.0


@torch.no_grad()
def validate():
    model.eval()

    hits_05 = 0   # R@1 IoU ≥ 0.5
    hits_07 = 0   # R@1 IoU ≥ 0.7
    total   = 0
    val_loss = 0.0

    for batch in val_loader:
        frame_embs, timestamps, query_emb = encode_batch(batch)
        start_logits, end_logits, confidence = model(frame_embs, timestamps, query_emb)

        B = start_logits.shape[0]
        N = start_logits.shape[1]

        for i in range(B):
            dur_i    = batch['duration'][i].item()
            n_frames = len(batch['frames'][i])   # real N for this sample

            # Decode predicted span
            s_idx, e_idx = decode_best_span(start_logits[i], end_logits[i])
            # Clamp to real frames
            s_idx = min(s_idx, n_frames - 1)
            e_idx = min(e_idx, n_frames - 1)
            pred_start, pred_end = to_seconds(s_idx, e_idx, n_frames, dur_i)

            # Ground truth in seconds
            ts_i = batch['timestamps'][i]
            gt_s_idx = batch['gt_start_idx'][i].item()
            gt_e_idx = batch['gt_end_idx'][i].item()
            gt_start_sec = ts_i[min(gt_s_idx, len(ts_i)-1)]
            gt_end_sec   = ts_i[min(gt_e_idx, len(ts_i)-1)]

            iou = temporal_iou(pred_start, pred_end, gt_start_sec, gt_end_sec)
            if iou >= 0.5:
                hits_05 += 1
            if iou >= 0.7:
                hits_07 += 1
            total += 1

        # Quick val loss (positive samples only)
        pos = ~batch['is_negative'].to(DEVICE)
        if pos.any():
            loss, *_ = combined_loss(
                start_logits[pos], end_logits[pos],
                batch['gt_start_idx'].to(DEVICE)[pos],
                batch['gt_end_idx'].to(DEVICE)[pos],
                query_emb[pos], frame_embs[pos],
            )
            val_loss += loss.item()

    r1_05 = hits_05 / total
    r1_07 = hits_07 / total

    return {
        'val/r1_iou05': r1_05,
        'val/r1_iou07': r1_07,
        'val/loss':     val_loss / len(val_loader),
    }


## Training Loop


In [ ]:
# ── Cell 17: Training loop ────────────────────────────────────────────────────
import wandb

run = wandb.init(
    project='groundzero',
    name=f'train-siglip2-lora-r{CFG["lora_rank"]}',
    config=CFG,
)

best_r1_05   = 0.0
global_step  = 0
ckpt_dir     = Path(CFG['ckpt_dir'])

for epoch in range(1, CFG['epochs'] + 1):
    epoch_loss = 0.0
    n_batches  = 0

    for batch in train_loader:
        metrics = train_step(batch)
        epoch_loss += metrics['train/loss']
        n_batches  += 1
        global_step += 1

        if global_step % CFG['log_every'] == 0:
            wandb.log(metrics, step=global_step)
            print(
                f'ep {epoch:02d}  step {global_step:05d}  '
                f'loss={metrics["train/loss"]:.4f}  '
                f'span={metrics["train/span_loss"]:.4f}  '
                f'iou={metrics["train/iou_loss"]:.4f}  '
                f'lr={metrics["train/lr"]:.2e}'
            )

    avg_loss = epoch_loss / n_batches
    print(f'--- Epoch {epoch:02d} avg loss: {avg_loss:.4f} ---')

    # Validate every val_every epochs
    if epoch % CFG['val_every'] == 0:
        val_metrics = validate()
        wandb.log({**val_metrics, 'epoch': epoch}, step=global_step)
        print(
            f'  Val → R@1 IoU=0.5: {val_metrics["val/r1_iou05"]:.4f}  '
            f'R@1 IoU=0.7: {val_metrics["val/r1_iou07"]:.4f}'
        )

        # Save best checkpoint
        if val_metrics['val/r1_iou05'] > best_r1_05:
            best_r1_05 = val_metrics['val/r1_iou05']
            ckpt_path  = ckpt_dir / f'best_epoch{epoch:02d}_r105_{best_r1_05:.4f}.pt'

            torch.save({
                'epoch':      epoch,
                'step':       global_step,
                'model':      model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'scheduler':  scheduler.state_dict(),
                'r1_iou05':   best_r1_05,
                'cfg':        CFG,
            }, ckpt_path)
            print(f'  Checkpoint saved → {ckpt_path.name}')

            wandb.save(str(ckpt_path))

run.finish()
print(f'Training complete. Best R@1 IoU=0.5: {best_r1_05:.4f}')


## Push Checkpoint to HuggingFace Hub


In [ ]:
# ── Cell 18: Push best checkpoint to HF Hub ───────────────────────────────────
from huggingface_hub import HfApi

api = HfApi()

# Create model repo if it doesn't exist
api.create_repo(CFG['hf_repo'], repo_type='model', exist_ok=True)

# Upload best checkpoint
best_ckpt = max(ckpt_dir.glob('best_*.pt'), key=lambda p: p.stat().st_mtime)
api.upload_file(
    path_or_fileobj=str(best_ckpt),
    path_in_repo=best_ckpt.name,
    repo_id=CFG['hf_repo'],
    repo_type='model',
)
print(f'Uploaded {best_ckpt.name} to {CFG["hf_repo"]}')


## Resume from Checkpoint

If the Colab session disconnects mid-training, use this cell to resume.


In [ ]:
# ── Cell 19: Resume from checkpoint (optional) ────────────────────────────────
# Uncomment and set RESUME_CKPT to resume a previous run
#
# RESUME_CKPT = '/content/checkpoints/best_epoch05_r105_0.3210.pt'
#
# ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
# model.load_state_dict(ckpt['model'])
# optimizer.load_state_dict(ckpt['optimizer'])
# scheduler.load_state_dict(ckpt['scheduler'])
# start_epoch = ckpt['epoch'] + 1
# global_step = ckpt['step']
# best_r1_05  = ckpt['r1_iou05']
# print(f'Resumed from epoch {ckpt["epoch"]}, step {global_step}, best R@1={best_r1_05:.4f}')
